# 🎰 Module 4.1: Multi-Armed Bandits for Image Processing

## Exploration vs Exploitation — Choosing the Best Image Filter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_04_Basic_RL_Algorithms/01_Multi_Armed_Bandits/multi_armed_bandits.ipynb)

---

## 🎯 Learning Objectives
1. Formalize the multi-armed bandit problem mathematically
2. Derive and implement **Epsilon-Greedy**, **UCB1**, and **Thompson Sampling** strategies
3. Apply bandits to selecting the best image filter from a set of candidates
4. Compare strategies via regret curves and reward analysis
5. Understand regret bounds and theoretical guarantees

---

In [ ]:
# Setup - Install dependencies (Google Colab)
!pip install numpy matplotlib opencv-python-headless pillow scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset
import torchvision
import numpy as np
from PIL import Image

# CIFAR-10 for image filter selection environment
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
# Take small subset for fast RL experiments
real_images = [np.array(cifar10[i][0]) for i in range(200)]
print(f"✅ CIFAR-10 loaded: Using {len(real_images)} real photos for RL environment")
print(f"   Image shape: {real_images[0].shape}")
print(f"   These will be our 'states' that the RL agent processes!")

## Deep Derivation: Bandit Algorithms and Regret Bounds

### Step 1: Formal Bandit Problem
$K$ arms with unknown reward distributions. Arm $i$ has mean $\mu_i$. Optimal arm: $i^* = \arg\max_i \mu_i$ with $\mu^* = \mu_{i^*}$.

**Regret** after $T$ steps:
$$\text{Regret}(T) = T\mu^* - \sum_{t=1}^T \mu_{A_t} = \sum_{i=1}^K \Delta_i \cdot E[N_i(T)]$$

where $\Delta_i = \mu^* - \mu_i$ is the suboptimality gap and $N_i(T)$ is the number of times arm $i$ is pulled.

### Step 2: Epsilon-Greedy — Regret Analysis

**Strategy:** With probability $\epsilon$, explore (uniform random); with probability $1-\epsilon$, exploit (best empirical arm).

**Theorem:** $\epsilon$-greedy with constant $\epsilon$ has **linear** regret: $\text{Regret}(T) = O(\epsilon T)$.

**Proof:**
At each step, probability of pulling a suboptimal arm $\geq \epsilon/K$. Expected regret contribution per step from exploration:
$$E[\text{regret per step}] \geq \epsilon \cdot \frac{1}{K}\sum_{i \neq i^*} \Delta_i$$

Summing over $T$ steps: $\text{Regret}(T) \geq \epsilon T \cdot \frac{1}{K}\sum_{i \neq i^*}\Delta_i = \Theta(\epsilon T)$ $\blacksquare$

**Fix:** Use decaying $\epsilon_t = \min(1, cK/(d^2 t))$ → regret becomes $O(\log T)$.

### Step 3: UCB1 — Derivation from Hoeffding's Inequality

**Hoeffding's Inequality:** For i.i.d. bounded r.v.s $X_1, \ldots, X_n \in [0, 1]$:
$$P\left(\bar{X}_n - \mu \geq t\right) \leq e^{-2nt^2}$$

**Derivation of UCB1:**
We want: $P(\mu_i > \hat{\mu}_i + U_i) \leq \delta$ for some small $\delta$.

By Hoeffding: $P(\mu_i > \hat{\mu}_i + t) \leq e^{-2N_i t^2}$

Set $\delta = e^{-2N_i t^2}$, solve for $t$:
$$t = \sqrt{\frac{\ln(1/\delta)}{2N_i}}$$

Choose $\delta = 1/t^4$ (ensures summability for union bound over time):
$$\text{UCB}_i(t) = \hat{\mu}_i + \sqrt{\frac{2\ln t}{N_i(t)}} \quad \blacksquare$$

### Step 4: UCB1 Regret Bound (Proof Sketch)

**Theorem (Auer et al., 2002):** UCB1 achieves:
$$\text{Regret}(T) \leq \sum_{i: \Delta_i > 0} \left(\frac{8\ln T}{\Delta_i} + (1 + \frac{\pi^2}{3})\Delta_i\right) = O\left(\sum_i \frac{\ln T}{\Delta_i}\right)$$

**Proof sketch:**
Arm $i$ is pulled at time $t$ only if $\text{UCB}_i(t) \geq \text{UCB}_{i^*}(t)$, meaning:
$$\hat{\mu}_i + \sqrt{\frac{2\ln t}{N_i}} \geq \hat{\mu}_{i^*} + \sqrt{\frac{2\ln t}{N_{i^*}}}$$

For this to happen often, either $\hat{\mu}_i$ overestimates $\mu_i$ or $\hat{\mu}_{i^*}$ underestimates $\mu^*$. By Hoeffding, both are rare events (probability $\sim 1/t^4$). Counting expected pulls: $E[N_i(T)] \leq \frac{8\ln T}{\Delta_i^2} + 1 + \frac{\pi^2}{3}$. $\blacksquare$

### Step 5: Thompson Sampling — Bayesian Derivation

**Prior:** $\mu_i \sim \text{Beta}(\alpha_i, \beta_i)$, initialized as $\text{Beta}(1, 1) = \text{Uniform}[0,1]$.

**Posterior update** after observing reward $r \in \{0, 1\}$ from arm $i$:
$$\text{Beta}(\alpha_i + r, \beta_i + (1 - r))$$

**Proof that Beta is the conjugate prior for Bernoulli:**
$$P(\mu | \text{data}) \propto P(\text{data} | \mu) \cdot P(\mu) = \mu^s(1-\mu)^f \cdot \mu^{\alpha-1}(1-\mu)^{\beta-1} = \mu^{\alpha+s-1}(1-\mu)^{\beta+f-1}$$
This is $\text{Beta}(\alpha + s, \beta + f)$ where $s$ = successes, $f$ = failures. $\blacksquare$

### Step 6: Lower Bound — No Algorithm Can Do Better Than $O(\log T)$

**Lai-Robbins Theorem (1985):** For any consistent strategy:
$$\liminf_{T \to \infty} \frac{E[N_i(T)]}{\ln T} \geq \frac{1}{\text{KL}(\mu_i \| \mu^*)}$$

This means UCB1 is **order-optimal** — logarithmic regret is the best possible!

### RL Connection: Bandits for Image Filter Selection
Given $K$ image filters, the bandit problem selects the best filter for a given image:
- Each "arm" = a filter (Gaussian blur, bilateral, median, etc.)
- Reward = image quality metric (PSNR, SSIM) after applying the filter
- The agent learns which filter works best WITHOUT trying all filters on every image

## 1. The Multi-Armed Bandit Problem

### 1.1 Formal Definition

A **$K$-armed bandit** is defined by the tuple $(\mathcal{A}, \{R_a\}_{a \in \mathcal{A}})$ where:

- $\mathcal{A} = \{1, 2, \ldots, K\}$ is the set of **arms** (actions)
- $R_a$ is the **reward distribution** associated with arm $a$
- At each time step $t = 1, 2, \ldots, T$, the agent selects an arm $A_t \in \mathcal{A}$ and receives reward $R_t \sim R_{A_t}$

### 1.2 The Objective

Maximize the **expected cumulative reward**:

$$\max \; \mathbb{E}\left[\sum_{t=1}^{T} R_t\right]$$

Equivalently, minimize the **cumulative regret**:

$$L_T = \mathbb{E}\left[\sum_{t=1}^{T} \left(\mu^* - \mu_{A_t}\right)\right] = T\mu^* - \sum_{t=1}^{T} \mathbb{E}[\mu_{A_t}]$$

where $\mu_a = \mathbb{E}[R_a]$ is the true mean reward of arm $a$, and $\mu^* = \max_a \mu_a$ is the optimal mean reward.

### 1.3 The Exploration-Exploitation Dilemma

- **Exploitation**: Choose the arm with the highest estimated reward $\rightarrow$ maximizes short-term gain
- **Exploration**: Try less-known arms $\rightarrow$ improves estimates, potentially finds better arms

### 1.4 Action-Value Estimation

The **sample-average** estimate of $Q(a)$ at time $t$:

$$Q_t(a) = \frac{\sum_{i=1}^{t-1} R_i \cdot \mathbb{1}[A_i = a]}{\sum_{i=1}^{t-1} \mathbb{1}[A_i = a]} = \frac{\text{total reward from arm } a}{\text{number of times arm } a \text{ was pulled}}$$

By the **law of large numbers**, $Q_t(a) \xrightarrow{a.s.} \mu_a$ as $N_t(a) \to \infty$.

### 1.5 Incremental Update Rule

We can update $Q$ incrementally:

$$Q_{n+1}(a) = Q_n(a) + \frac{1}{n}\left[R_n - Q_n(a)\right]$$

Or with a fixed step size $\alpha$:

$$Q_{n+1}(a) = Q_n(a) + \alpha\left[R_n - Q_n(a)\right]$$

## 2. Image Filter Bandit Setup

### Our Scenario

We have a **degraded image** and $K$ candidate filters. Each filter produces a different enhancement. The **reward** is the quality improvement, measured by a sharpness metric (variance of the Laplacian).

Each arm $a$ corresponds to one image filter, and pulling the arm means applying that filter. The reward has some stochasticity because we add small random noise to model real-world variance.

In [ ]:
def create_test_image(size=256):
    """Create a synthetic test image with various features."""
    img = np.zeros((size, size), dtype=np.float64)
    
    # Gradient background
    for i in range(size):
        img[i, :] = i / size * 0.3
    
    # Circles
    for cx, cy, r, val in [(80, 80, 30, 0.9), (180, 180, 40, 0.7), (80, 180, 25, 0.5)]:
        Y, X = np.ogrid[:size, :size]
        mask = (X - cx)**2 + (Y - cy)**2 <= r**2
        img[mask] = val
    
    # Rectangles
    img[30:70, 140:220] = 0.8
    img[150:170, 50:120] = 0.6
    
    # Edges / lines
    img[120, 20:240] = 1.0
    img[20:240, 128] = 1.0
    
    return img

def degrade_image(img, noise_level=0.15):
    """Add Gaussian noise and slight blur to simulate degradation."""
    blurred = cv2.GaussianBlur(img, (5, 5), 1.0)
    noisy = blurred + noise_level * np.random.randn(*blurred.shape)
    return np.clip(noisy, 0, 1)

def sharpness_score(img):
    """Variance of Laplacian — higher means sharper."""
    img_uint8 = (img * 255).astype(np.uint8)
    return cv2.Laplacian(img_uint8, cv2.CV_64F).var()

# Define image filters (the arms)
FILTER_NAMES = [
    'Gaussian Blur (3x3)',
    'Sharpen',
    'Edge Enhance',
    'Median Filter (3)',
    'Bilateral Filter',
    'Histogram Eq',
    'Unsharp Mask',
    'Gaussian Blur (7x7)'
]

def apply_filter(img, filter_idx):
    """Apply the filter specified by filter_idx."""
    img_uint8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    
    if filter_idx == 0:  # Gaussian blur 3x3
        result = cv2.GaussianBlur(img_uint8, (3, 3), 0.8)
    elif filter_idx == 1:  # Sharpen
        kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
        result = cv2.filter2D(img_uint8, -1, kernel)
    elif filter_idx == 2:  # Edge enhance
        kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
        result = cv2.filter2D(img_uint8, -1, kernel)
    elif filter_idx == 3:  # Median filter
        result = cv2.medianBlur(img_uint8, 3)
    elif filter_idx == 4:  # Bilateral filter
        result = cv2.bilateralFilter(img_uint8, 5, 75, 75)
    elif filter_idx == 5:  # Histogram equalization
        result = cv2.equalizeHist(img_uint8)
    elif filter_idx == 6:  # Unsharp mask
        blurred = cv2.GaussianBlur(img_uint8, (0, 0), 2.0)
        result = cv2.addWeighted(img_uint8, 1.5, blurred, -0.5, 0)
    elif filter_idx == 7:  # Gaussian blur 7x7
        result = cv2.GaussianBlur(img_uint8, (7, 7), 2.0)
    else:
        result = img_uint8
    
    return result.astype(np.float64) / 255.0

# Create test images
clean_img = create_test_image()
degraded_img = degrade_image(clean_img)

# Show the images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(clean_img, cmap='gray')
axes[0].set_title(f'Clean Image\nSharpness: {sharpness_score(clean_img):.1f}')
axes[0].axis('off')
axes[1].imshow(degraded_img, cmap='gray')
axes[1].set_title(f'Degraded Image\nSharpness: {sharpness_score(degraded_img):.1f}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
class ImageFilterBandit:
    """Multi-armed bandit where each arm is an image filter."""
    
    def __init__(self, degraded_img, noise_std=0.05):
        self.degraded_img = degraded_img
        self.base_sharpness = sharpness_score(degraded_img)
        self.noise_std = noise_std
        self.K = len(FILTER_NAMES)
        
        # Precompute true mean rewards
        self.true_means = np.zeros(self.K)
        for a in range(self.K):
            filtered = apply_filter(degraded_img, a)
            self.true_means[a] = sharpness_score(filtered) - self.base_sharpness
        
        # Normalize to [0, 1]
        self.reward_range = max(self.true_means.max() - self.true_means.min(), 1e-8)
        self.true_means_norm = (self.true_means - self.true_means.min()) / self.reward_range
        self.optimal_arm = np.argmax(self.true_means_norm)
        self.mu_star = self.true_means_norm[self.optimal_arm]
    
    def pull(self, arm):
        """Pull an arm and get a stochastic reward."""
        reward = self.true_means_norm[arm] + self.noise_std * np.random.randn()
        return np.clip(reward, 0, 1)

bandit = ImageFilterBandit(degraded_img)

# Display true reward distribution
fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.Set2(np.linspace(0, 1, bandit.K))
bars = ax.bar(range(bandit.K), bandit.true_means_norm, color=colors, edgecolor='black')
ax.set_xticks(range(bandit.K))
ax.set_xticklabels(FILTER_NAMES, rotation=30, ha='right')
ax.set_ylabel('Normalized Mean Reward')
ax.set_title('True Mean Rewards of Image Filter Arms (Unknown to Agent)')
ax.axhline(y=bandit.mu_star, color='red', linestyle='--', label=f'Optimal: {FILTER_NAMES[bandit.optimal_arm]}')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nOptimal arm: {FILTER_NAMES[bandit.optimal_arm]} (μ* = {bandit.mu_star:.4f})")

## 3. Epsilon-Greedy Strategy

### 3.1 Algorithm

The $\epsilon$-greedy policy balances exploration and exploitation with a simple rule:

$$A_t = \begin{cases} \arg\max_a Q_t(a) & \text{with probability } 1-\epsilon \quad (\text{exploit}) \\ \text{Uniform}(\mathcal{A}) & \text{with probability } \epsilon \quad (\text{explore}) \end{cases}$$

### 3.2 Analysis

**Probability of choosing the optimal arm:**

$$P(A_t = a^*) \geq 1 - \epsilon + \frac{\epsilon}{K}$$

**Expected per-step regret:**

$$\mathbb{E}[\mu^* - \mu_{A_t}] \leq \epsilon \cdot \mu^* + (1-\epsilon) \cdot 0 = \epsilon \mu^*$$

More precisely, the cumulative regret for $\epsilon$-greedy satisfies:

$$L_T = O(\epsilon T + K/\epsilon)$$

Setting $\epsilon = \sqrt{K/T}$ gives $L_T = O(\sqrt{KT})$.

### 3.3 Decaying $\epsilon$

For better long-run performance, decay $\epsilon$ over time:

$$\epsilon_t = \min\left(1, \frac{cK}{d^2 t}\right)$$

where $c > 0$ and $d = \min_{a: \mu_a < \mu^*}(\mu^* - \mu_a)$ is the minimum sub-optimality gap.

In [ ]:
class EpsilonGreedy:
    """Epsilon-greedy bandit agent."""
    
    def __init__(self, K, epsilon=0.1, decay=False):
        self.K = K
        self.epsilon_init = epsilon
        self.decay = decay
        self.reset()
    
    def reset(self):
        self.Q = np.zeros(self.K)
        self.N = np.zeros(self.K)
        self.t = 0
    
    @property
    def epsilon(self):
        if self.decay:
            return self.epsilon_init / (1 + self.t * 0.001)
        return self.epsilon_init
    
    def select_arm(self):
        self.t += 1
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.K)
        else:
            return np.argmax(self.Q)
    
    def update(self, arm, reward):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

def run_experiment(bandit, agent, T=2000):
    """Run a bandit experiment and collect metrics."""
    rewards = np.zeros(T)
    regrets = np.zeros(T)
    arms_chosen = np.zeros(T, dtype=int)
    
    for t in range(T):
        arm = agent.select_arm()
        reward = bandit.pull(arm)
        agent.update(arm, reward)
        
        rewards[t] = reward
        regrets[t] = bandit.mu_star - bandit.true_means_norm[arm]
        arms_chosen[t] = arm
    
    return rewards, regrets, arms_chosen

# Run epsilon-greedy with different epsilon values
T = 3000
epsilons = [0.01, 0.1, 0.3]
results_eg = {}

for eps in epsilons:
    all_rewards, all_regrets = [], []
    for run in range(50):
        agent = EpsilonGreedy(bandit.K, epsilon=eps)
        rewards, regrets, arms = run_experiment(bandit, agent, T)
        all_rewards.append(rewards)
        all_regrets.append(regrets)
    results_eg[eps] = {
        'rewards': np.mean(all_rewards, axis=0),
        'cum_regret': np.cumsum(np.mean(all_regrets, axis=0))
    }

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for eps in epsilons:
    axes[0].plot(np.cumsum(results_eg[eps]['rewards']) / (np.arange(T) + 1),
                 label=f'ε = {eps}')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Average Reward')
axes[0].set_title('ε-Greedy: Average Reward Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for eps in epsilons:
    axes[1].plot(results_eg[eps]['cum_regret'], label=f'ε = {eps}')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Cumulative Regret')
axes[1].set_title('ε-Greedy: Cumulative Regret')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Upper Confidence Bound (UCB)

### 4.1 Motivation: Optimism in the Face of Uncertainty

Instead of exploring randomly, UCB explores **systematically** — it favors arms whose mean reward estimate has high **uncertainty**.

### 4.2 Hoeffding's Inequality

For $n$ i.i.d. random variables $X_1, \ldots, X_n$ with $X_i \in [0,1]$, let $\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$. Then:

$$P\left(\bar{X}_n - \mathbb{E}[\bar{X}_n] \geq \epsilon\right) \leq e^{-2n\epsilon^2}$$

### 4.3 Deriving the UCB1 Bound

We want to construct an **upper confidence bound** $U_t(a)$ such that $\mu_a \leq U_t(a)$ with high probability.

**Step 1:** Apply Hoeffding to arm $a$ with $n = N_t(a)$ pulls:

$$P\left(Q_t(a) - \mu_a \geq \epsilon\right) \leq e^{-2N_t(a)\epsilon^2}$$

**Step 2:** Set the right side to $\delta = t^{-2c}$ (decreasing with time):

$$e^{-2N_t(a)\epsilon^2} = t^{-2c} \implies \epsilon = \sqrt{\frac{c \ln t}{N_t(a)}}$$

**Step 3:** The UCB1 formula (with $c = 1$):

$$A_t = \arg\max_a \left[ Q_t(a) + \sqrt{\frac{2 \ln t}{N_t(a)}} \right]$$

The term $\sqrt{\frac{2\ln t}{N_t(a)}}$ is the **confidence bonus**: large when arm $a$ has been pulled few times (high uncertainty).

### 4.4 Regret Bound

**Theorem (Auer et al., 2002):** The cumulative regret of UCB1 satisfies:

$$L_T \leq \sum_{a: \mu_a < \mu^*} \left( \frac{8 \ln T}{\Delta_a} + (1 + \frac{\pi^2}{3}) \Delta_a \right)$$

where $\Delta_a = \mu^* - \mu_a$ is the **sub-optimality gap**.

This gives $L_T = O\left(\sqrt{KT \ln T}\right)$ — **logarithmic** in $T$ for the gap-dependent bound!

In [ ]:
class UCB:
    """Upper Confidence Bound (UCB1) agent."""
    
    def __init__(self, K, c=2.0):
        self.K = K
        self.c = c
        self.reset()
    
    def reset(self):
        self.Q = np.zeros(self.K)
        self.N = np.zeros(self.K)
        self.t = 0
    
    def select_arm(self):
        self.t += 1
        # Pull each arm once first
        for a in range(self.K):
            if self.N[a] == 0:
                return a
        
        ucb_values = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        return np.argmax(ucb_values)
    
    def update(self, arm, reward):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

# Visualize UCB values over time
agent_ucb_demo = UCB(bandit.K, c=2.0)
ucb_history = []

for t in range(500):
    arm = agent_ucb_demo.select_arm()
    reward = bandit.pull(arm)
    agent_ucb_demo.update(arm, reward)
    
    if t >= bandit.K:  # After initial round-robin
        ucb_vals = agent_ucb_demo.Q + 2.0 * np.sqrt(np.log(agent_ucb_demo.t) / np.maximum(agent_ucb_demo.N, 1))
        ucb_history.append(ucb_vals.copy())

ucb_history = np.array(ucb_history)

fig, ax = plt.subplots(figsize=(14, 6))
for a in range(bandit.K):
    ax.plot(ucb_history[:, a], alpha=0.7, label=FILTER_NAMES[a])
    ax.axhline(y=bandit.true_means_norm[a], color=plt.cm.tab10(a), linestyle='--', alpha=0.3)

ax.set_xlabel('Time Step (after initialization)')
ax.set_ylabel('UCB Value')
ax.set_title('UCB Values Converging Over Time\n(Dashed lines = true means)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Thompson Sampling

### 5.1 The Bayesian Approach

Thompson Sampling maintains a **posterior distribution** over the mean reward of each arm and samples from it to make decisions.

### 5.2 For Bernoulli Bandits: Beta-Bernoulli Model

If rewards are binary ($R_t \in \{0, 1\}$), use a **Beta** prior:

**Prior:** $\mu_a \sim \text{Beta}(\alpha_a, \beta_a)$ with $\alpha_a = \beta_a = 1$ (uniform)

**Posterior after observing $S_a$ successes and $F_a$ failures:**

$$\mu_a \mid \text{data} \sim \text{Beta}(\alpha_a + S_a, \; \beta_a + F_a)$$

This is a **conjugate update** — the posterior stays in the Beta family.

### 5.3 For Gaussian Bandits: Normal-Normal Model

If rewards are continuous with known variance $\sigma^2$, use a **Gaussian** prior:

**Prior:** $\mu_a \sim \mathcal{N}(\mu_0, \sigma_0^2)$

**Posterior after $n$ observations with sample mean $\bar{x}$:**

$$\mu_a \mid \text{data} \sim \mathcal{N}\left(\frac{\sigma^2 \mu_0 + n \sigma_0^2 \bar{x}}{\sigma^2 + n\sigma_0^2}, \; \frac{\sigma^2 \sigma_0^2}{\sigma^2 + n \sigma_0^2}\right)$$

As $n \to \infty$, the posterior concentrates on the true mean.

### 5.4 Algorithm

At each step $t$:

1. For each arm $a$, sample $\tilde{\mu}_a \sim P(\mu_a \mid \text{history})$
2. Select $A_t = \arg\max_a \tilde{\mu}_a$
3. Observe reward $R_t$ and update posterior of arm $A_t$

### 5.5 Regret Bound

**Theorem (Agrawal & Goyal, 2012):** For Bernoulli bandits, Thompson Sampling achieves:

$$L_T = O\left(\sum_{a: \mu_a < \mu^*} \frac{\ln T}{\Delta_a}\right)$$

This matches the **Lai-Robbins lower bound** asymptotically, making Thompson Sampling **asymptotically optimal**!

In [ ]:
class ThompsonSamplingGaussian:
    """Thompson Sampling with Gaussian prior/posterior for continuous rewards."""
    
    def __init__(self, K, prior_mean=0.5, prior_var=1.0, noise_var=0.05**2):
        self.K = K
        self.prior_mean = prior_mean
        self.prior_var = prior_var
        self.noise_var = noise_var
        self.reset()
    
    def reset(self):
        self.mu = np.full(self.K, self.prior_mean)
        self.var = np.full(self.K, self.prior_var)
        self.N = np.zeros(self.K)
        self.sum_rewards = np.zeros(self.K)
    
    def select_arm(self):
        samples = np.array([np.random.normal(self.mu[a], np.sqrt(self.var[a]))
                           for a in range(self.K)])
        return np.argmax(samples)
    
    def update(self, arm, reward):
        self.N[arm] += 1
        self.sum_rewards[arm] += reward
        
        n = self.N[arm]
        x_bar = self.sum_rewards[arm] / n
        
        # Bayesian update: Normal-Normal conjugate
        prior_precision = 1.0 / self.prior_var
        data_precision = n / self.noise_var
        
        self.var[arm] = 1.0 / (prior_precision + data_precision)
        self.mu[arm] = self.var[arm] * (prior_precision * self.prior_mean + data_precision * x_bar)

# Visualize posterior evolution
ts_demo = ThompsonSamplingGaussian(bandit.K)
snapshots = [10, 50, 200, 500]

fig, axes = plt.subplots(1, len(snapshots), figsize=(20, 4))
x_range = np.linspace(-0.5, 1.5, 300)

for step in range(max(snapshots) + 1):
    arm = ts_demo.select_arm()
    reward = bandit.pull(arm)
    ts_demo.update(arm, reward)
    
    if step + 1 in snapshots:
        ax_idx = snapshots.index(step + 1)
        ax = axes[ax_idx]
        for a in range(min(bandit.K, 5)):  # Show first 5 arms for clarity
            pdf = (1 / np.sqrt(2 * np.pi * ts_demo.var[a])) * \
                  np.exp(-0.5 * (x_range - ts_demo.mu[a])**2 / ts_demo.var[a])
            ax.plot(x_range, pdf, label=FILTER_NAMES[a][:12], alpha=0.8)
            ax.axvline(x=bandit.true_means_norm[a], linestyle=':', alpha=0.3,
                       color=plt.cm.tab10(a))
        ax.set_title(f'After {step+1} Steps')
        ax.set_xlabel('μ')
        if ax_idx == 0:
            ax.set_ylabel('Posterior Density')
        if ax_idx == len(snapshots) - 1:
            ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Thompson Sampling: Posterior Distributions Converging to True Means', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Comparing All Strategies

Let's run all three algorithms on the same image filter bandit and compare their performance across multiple runs.

In [ ]:
T = 3000
n_runs = 100

agents_config = {
    'ε-Greedy (ε=0.1)': lambda: EpsilonGreedy(bandit.K, epsilon=0.1),
    'ε-Greedy (decay)': lambda: EpsilonGreedy(bandit.K, epsilon=0.5, decay=True),
    'UCB1 (c=2)': lambda: UCB(bandit.K, c=2.0),
    'Thompson Sampling': lambda: ThompsonSamplingGaussian(bandit.K),
}

all_results = {}
for name, agent_fn in agents_config.items():
    run_rewards = np.zeros((n_runs, T))
    run_regrets = np.zeros((n_runs, T))
    run_arms = np.zeros((n_runs, T), dtype=int)
    
    for run in range(n_runs):
        agent = agent_fn()
        rewards, regrets, arms = run_experiment(bandit, agent, T)
        run_rewards[run] = rewards
        run_regrets[run] = regrets
        run_arms[run] = arms
    
    all_results[name] = {
        'rewards': run_rewards,
        'regrets': run_regrets,
        'arms': run_arms
    }

print("✅ All experiments complete!")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']

# 1. Cumulative Regret
ax = axes[0, 0]
for i, (name, res) in enumerate(all_results.items()):
    cum_regret = np.cumsum(res['regrets'], axis=1)
    mean_cr = cum_regret.mean(axis=0)
    std_cr = cum_regret.std(axis=0)
    ax.plot(mean_cr, label=name, color=colors[i], linewidth=2)
    ax.fill_between(range(T), mean_cr - std_cr, mean_cr + std_cr,
                    alpha=0.1, color=colors[i])
ax.set_xlabel('Time Step')
ax.set_ylabel('Cumulative Regret')
ax.set_title('Cumulative Regret Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Average Reward
ax = axes[0, 1]
for i, (name, res) in enumerate(all_results.items()):
    avg_reward = np.cumsum(res['rewards'], axis=1) / (np.arange(T) + 1)
    ax.plot(avg_reward.mean(axis=0), label=name, color=colors[i], linewidth=2)
ax.axhline(y=bandit.mu_star, color='black', linestyle='--', alpha=0.5, label='μ*')
ax.set_xlabel('Time Step')
ax.set_ylabel('Average Reward')
ax.set_title('Average Reward Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Optimal Arm Selection %
ax = axes[1, 0]
window = 100
for i, (name, res) in enumerate(all_results.items()):
    optimal_pct = (res['arms'] == bandit.optimal_arm).astype(float).mean(axis=0)
    smoothed = np.convolve(optimal_pct, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=name, color=colors[i], linewidth=2)
ax.set_xlabel('Time Step')
ax.set_ylabel('% Optimal Arm Selected')
ax.set_title('Optimal Arm Selection Rate (Smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Arm Selection Frequency (final)
ax = axes[1, 1]
width = 0.2
x_pos = np.arange(bandit.K)
for i, (name, res) in enumerate(all_results.items()):
    arm_counts = np.bincount(res['arms'].flatten(), minlength=bandit.K)
    arm_freq = arm_counts / arm_counts.sum()
    ax.bar(x_pos + i * width, arm_freq, width, label=name, color=colors[i], alpha=0.8)
ax.set_xticks(x_pos + 1.5 * width)
ax.set_xticklabels([f[:10] for f in FILTER_NAMES], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Selection Frequency')
ax.set_title('Arm Selection Distribution')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Multi-Armed Bandit Strategy Comparison on Image Filters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Theoretical Analysis Summary

### 7.1 Regret Bounds Comparison

| Algorithm | Gap-Dependent Bound | Gap-Independent Bound | Optimal? |
|-----------|--------------------|-----------------------|----------|
| $\epsilon$-Greedy (fixed) | $O(\epsilon T)$ | $O(\epsilon T)$ | No (linear) |
| $\epsilon$-Greedy (decay) | $O(K \ln T / \Delta_{\min})$ | $O(\sqrt{KT})$ | Nearly |
| UCB1 | $O\left(\sum_a \frac{\ln T}{\Delta_a}\right)$ | $O(\sqrt{KT \ln T})$ | Nearly |
| Thompson Sampling | $O\left(\sum_a \frac{\ln T}{\Delta_a}\right)$ | $O(\sqrt{KT \ln K})$ | Yes (asymptotic) |

### 7.2 Lai-Robbins Lower Bound

No algorithm can achieve regret lower than:

$$\liminf_{T \to \infty} \frac{L_T}{\ln T} \geq \sum_{a: \mu_a < \mu^*} \frac{\Delta_a}{\text{KL}(R_a \| R_{a^*})}$$

where $\text{KL}(\cdot \| \cdot)$ is the Kullback-Leibler divergence between reward distributions.

### 7.3 Key Takeaways

- **$\epsilon$-Greedy**: Simple but suffers linear regret with fixed $\epsilon$; decaying $\epsilon$ helps
- **UCB**: Deterministic exploration with strong guarantees; slightly conservative
- **Thompson Sampling**: Bayesian, empirically excellent, asymptotically optimal; requires specifying a prior

In [ ]:
# Show the best filter selected by each strategy
print("=" * 60)
print("FINAL RESULTS: Best Filter Selected by Each Strategy")
print("=" * 60)
print(f"\nTrue best filter: {FILTER_NAMES[bandit.optimal_arm]}")
print(f"True best reward: {bandit.mu_star:.4f}\n")

for name, res in all_results.items():
    # Most selected arm in last 500 steps
    last_arms = res['arms'][:, -500:].flatten()
    most_common = np.bincount(last_arms, minlength=bandit.K).argmax()
    total_reward = res['rewards'].sum(axis=1).mean()
    print(f"{name:25s} → {FILTER_NAMES[most_common]:20s} | Avg total reward: {total_reward:.1f}")

# Show what each filter does visually
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(bandit.K):
    row, col = i // 4, i % 4
    filtered = apply_filter(degraded_img, i)
    axes[row, col].imshow(filtered, cmap='gray')
    axes[row, col].set_title(f'{FILTER_NAMES[i]}\nReward: {bandit.true_means_norm[i]:.3f}',
                             fontsize=10)
    axes[row, col].axis('off')

plt.suptitle('All Image Filters (Arms) and Their Rewards', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 📝 Summary

In this notebook, we covered:

1. **Multi-Armed Bandit Formulation** — the formal framework $(\mathcal{A}, \{R_a\})$ with regret as the performance metric
2. **Epsilon-Greedy** — simple exploration via random actions with probability $\epsilon$; linear regret when fixed
3. **UCB1** — deterministic, optimistic exploration using confidence bounds from Hoeffding's inequality; $O(\sqrt{KT \ln T})$ regret
4. **Thompson Sampling** — Bayesian posterior sampling; asymptotically optimal with $O(\sqrt{KT})$ regret
5. **Image Filter Selection** — each arm maps to a real image filter; reward is measured by sharpness improvement

### 🔑 Key Equations

| Concept | Formula |
|---------|----------|
| Action-Value | $Q_t(a) = \frac{1}{N_t(a)} \sum_{i=1}^{t-1} R_i \mathbb{1}[A_i=a]$ |
| UCB1 | $A_t = \arg\max_a [Q_t(a) + c\sqrt{\ln t / N_t(a)}]$ |
| Regret | $L_T = T\mu^* - \sum_{t=1}^T \mathbb{E}[\mu_{A_t}]$ |

---

## ➡️ Next

In the next notebook, we'll explore **Dynamic Programming** methods — Policy Evaluation, Policy Iteration, and Value Iteration — and apply them to finding the optimal sequence of image processing operations.